In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Import our custom preprocessor function from earlier
import sys
sys.path.append('../')
from utils import preprocess_text  # We will save our preprocess function in utils.py!

# Load dataset (take 20,000 rows for fast baseline testing)
df = pd.read_csv('../data/quora_ques.csv').dropna(subset=['question1', 'question2'])
df_sample = df.head(20000).copy()

print("Preprocessing sample questions...")
df_sample['q1_clean'] = df_sample['question1'].apply(lambda x: preprocess_text(x))
df_sample['q2_clean'] = df_sample['question2'].apply(lambda x: preprocess_text(x))

print("Done! Rows ready:", len(df_sample))

Preprocessing sample questions...
Done! Rows ready: 20000


In [2]:
# Combine all questions to build vocabulary
all_questions = pd.concat([df_sample['q1_clean'], df_sample['q2_clean']])

# Initialize TF-IDF Vectorizer (limit max features to 10,000 top words)
tfidf = TfidfVectorizer(max_features=10000)
tfidf.fit(all_questions)

print("Vocabulary size:", len(tfidf.vocabulary_))

# Vectorize Question 1 and Question 2
q1_vecs = tfidf.transform(df_sample['q1_clean'])
q2_vecs = tfidf.transform(df_sample['q2_clean'])

# Compute row-by-row Cosine Similarity using dot product
# Since vectors are normalized, dot product between row i of Q1 and Q2 = Cosine Similarity
similarity_scores = np.array([
    cosine_similarity(q1_vecs[i], q2_vecs[i])[0][0]
    for i in range(len(df_sample))
])

df_sample['tfidf_similarity'] = similarity_scores
df_sample[['question1', 'question2', 'is_duplicate', 'tfidf_similarity']].head(10)

Vocabulary size: 10000


,question1,question2,is_duplicate,tfidf_similarity
0,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0,0.979012
1,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0,0.764914
2,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0,0.293843
3,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0,0.000000
4,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0,0.254222
5,Astrology: I am a Capricorn Sun Cap moon and c...,"I'm a triple Capricorn (Sun, Moon and ascendan...",1,0.482883
6,Should I buy tiago?,What keeps childern active and far from phone ...,0,0.000000
7,How can I be a good geologist?,What should I do to be a great geologist?,1,0.774202
8,When do you use シ instead of し?,"When do you use ""&"" instead of ""and""?",0,1.000000
9,Motorola (company): Can I hack my Charter Moto...,How do I hack Motorola DCX3400 for free internet?,0,0.699812


In [3]:
# Test TF-IDF on synonyms / rephrased questions
q1_test = preprocess_text("How can I lose weight fast?")
q2_test = preprocess_text("What is the best way to reduce body fat?")

v1 = tfidf.transform([q1_test])
v2 = tfidf.transform([q2_test])

sim = cosine_similarity(v1, v2)[0][0]

print("Question 1:", "How can I lose weight fast?")
print("Question 2:", "What is the best way to reduce body fat?")
print(f"TF-IDF Similarity Score: {sim:.4f} ({sim*100:.1f}%)")

Question 1: How can I lose weight fast?
Question 2: What is the best way to reduce body fat?
TF-IDF Similarity Score: 0.0000 (0.0%)
